# AyeAI Physical AI Platform - Distributed Compute / GPU Reliability Lab

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks.**  
Licensed under **GNU General Public License v3.0 (GPL-3.0-only)**.  
This notebook belongs to the separate `compute-orchestration` drop-in package. It is not part of Manufacturing and does not modify the canonical kernel.

## What this notebook does

It runs a Monte Carlo reliability estimate for a series system with independent exponentially distributed component lifetimes. It uses PyTorch CUDA when a compatible GPU is attached; otherwise it uses NumPy on CPU. It compares the Monte Carlo estimate with the closed-form reliability:

\[
R(t)=\exp\left(-t\sum_i \lambda_i\right)
\]

The analytic comparison makes this a falsifiable numerical benchmark, not just a GPU availability demo. The result is marked `SIMULATED`, never `VERIFIED` or `PHYSICAL_VERIFIED`.

Colab is an optional remote execution environment. The platform remains browser-only and static-first; no local desktop solver, daemon, or mandatory cloud service is introduced.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
import sys, platform, importlib.util, json, hashlib, time, math, os
print("Python",sys.version)
print("Platform",platform.platform())
for p in ("numpy","torch","jax","cupy"):
    print(p, "AVAILABLE" if importlib.util.find_spec(p) else "not installed")


## Load the job

Upload `reliability-series-job.json` from this package. If you do not upload a file, the notebook uses the documented sample below. The input digest is SHA-256 over canonical JSON of the `payload` object.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
from google.colab import files
uploaded = files.upload()  # Upload samples/reliability-series-job.json


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
def canonical_json(value):
    return json.dumps(value,sort_keys=True,separators=(",",":"),ensure_ascii=False)
if uploaded:
    selected = next(iter(uploaded))
    job = json.loads(uploaded[selected].decode("utf-8"))
else:
    payload = {"failureRatesPerHour":[0.0001,0.0002,0.00015],
               "missionTimeHours":1000,"samples":250000,"seed":4701}
    job = {"protocol":"ayeai.compute.job/v1","jobId":"reliability-demo-001",
           "capability":"reliability.series.monte-carlo","executionClass":"simulation",
           "resourceRequirements":{"accelerator":"preferred","minMemoryGiB":2,"maxRuntimeSeconds":120},
           "provenance":{"sourceRef":"sir://sample-system/mission-01",
                         "configurationRef":"config://sample-system/rev-1"},
           "payload":payload}
    job["inputDigest"]=hashlib.sha256(canonical_json(payload).encode()).hexdigest()
assert job["protocol"]=="ayeai.compute.job/v1"
assert job["capability"]=="reliability.series.monte-carlo"
assert job["executionClass"]=="simulation"
assert hashlib.sha256(canonical_json(job["payload"]).encode()).hexdigest()==job["inputDigest"]
p=job["payload"]
rates=p["failureRatesPerHour"]; mission=p["missionTimeHours"]; samples=p["samples"]; seed=p["seed"]
assert isinstance(rates,list) and len(rates)>0 and all(math.isfinite(x) and x>0 for x in rates)
assert math.isfinite(mission) and mission>0
assert isinstance(samples,int) and 1000<=samples<=5_000_000
assert isinstance(seed,int) and seed>=0
print("Validated job",job["jobId"],"components",len(rates),"samples",samples)


## Execute on GPU when available

PyTorch is used only for this GPU-capable Monte Carlo workload. It is not used to replace OR-Tools CP-SAT, HiGHS, or SimPy. If the GPU runtime is unavailable, the notebook executes the same model with NumPy CPU operations.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
import numpy as np
start=time.perf_counter()
engine="auto"  # Set to "torch", "jax", or "numpy" to force a supported path.
gpu_used=False
accelerator_used=False
device_name="CPU / NumPy"
backend_version=np.__version__
selected_engine="numpy-cpu"
if engine in ("auto","torch"):
    try:
        import torch
        if torch.cuda.is_available():
            torch.manual_seed(seed)
            device=torch.device("cuda")
            rate_tensor=torch.tensor(rates,dtype=torch.float64,device=device)
            u=torch.rand((samples,len(rates)),dtype=torch.float64,device=device).clamp_min(torch.finfo(torch.float64).tiny)
            system_lifetime=(-torch.log(u)/rate_tensor).min(dim=1).values
            success=(system_lifetime>mission).to(torch.float64)
            estimate=float(success.mean().item())
            se=float(torch.sqrt(success.mean()*(1-success.mean())/samples).item())
            device_name=torch.cuda.get_device_name(0)
            backend_version=torch.__version__
            gpu_used=True
            accelerator_used=True
            selected_engine="pytorch-cuda"
            del u,system_lifetime,success,rate_tensor
            torch.cuda.empty_cache()
        elif engine=="torch":
            raise RuntimeError("PyTorch CUDA requested but no CUDA device is available")
    except ImportError:
        if engine=="torch": raise
if not gpu_used and engine in ("auto","jax"):
    try:
        import jax
        import jax.numpy as jnp
        accelerator_devices=[d for d in jax.devices() if d.platform in ("gpu","tpu")]
        if accelerator_devices:
            key=jax.random.PRNGKey(seed)
            u=jax.random.uniform(key,(samples,len(rates)),minval=1e-7,maxval=1.0,dtype=jnp.float32)
            lifetimes=-jnp.log(u)/jnp.asarray(rates,dtype=jnp.float32)
            success=(jnp.min(lifetimes,axis=1)>mission).astype(jnp.float32)
            mean=jnp.mean(success)
            estimate=float(jax.device_get(mean))
            se=float(jax.device_get(jnp.sqrt(mean*(1-mean)/samples)))
            device_name=str(accelerator_devices[0])
            backend_version=jax.__version__
            gpu_used=(accelerator_devices[0].platform=="gpu")
            accelerator_used=True
            selected_engine="jax-accelerator"
        elif engine=="jax":
            raise RuntimeError("JAX GPU/TPU requested but no accelerator device is available")
    except ImportError:
        if engine=="jax": raise
if not accelerator_used:
    if engine not in ("auto","numpy"):
        raise RuntimeError("Requested accelerator backend is unavailable")
    rng=np.random.default_rng(seed)
    u=np.maximum(rng.random((samples,len(rates))),np.finfo(np.float64).tiny)
    system_lifetime=(-np.log(u)/np.asarray(rates,dtype=np.float64)).min(axis=1)
    success=(system_lifetime>mission)
    estimate=float(success.mean())
    se=float(math.sqrt(estimate*(1-estimate)/samples))
    selected_engine="numpy-cpu"
analytic=math.exp(-mission*sum(rates))
abs_error=abs(estimate-analytic)
tolerance=5.0*se+0.002
ci=[max(0.0,estimate-1.96*se),min(1.0,estimate+1.96*se)]
validation="PASS" if abs_error<=tolerance else "HOLD"
elapsed=time.perf_counter()-start
print({"engine":selected_engine,"backend_version":backend_version,"device":device_name,
       "gpu_used":gpu_used,"accelerator_used":accelerator_used,"samples":samples,"estimate":estimate,"analytic":analytic,
       "standard_error":se,"ci95":ci,"absolute_error":abs_error,
       "acceptance_tolerance":tolerance,"validation":validation})
assert validation=="PASS", "Monte Carlo estimate did not agree with analytical reliability within tolerance."


## Build a provenance-bearing result

The result carries request identity, input digest, provider/runtime details, output digest and explicit simulation evidence state. The digest is an integrity check, not a digital signature. A production deployment must add authenticated provider identity and signed provenance if required by its trust model.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
provider_id = "colab-pytorch-cuda" if selected_engine=="pytorch-cuda" else (
    "colab-jax-accelerator" if selected_engine=="jax-accelerator" else "colab-numpy-cpu")
provider_version=backend_version
result_payload={
 "model":"independent-exponential-series-system",
 "componentCount":len(rates),"missionTimeHours":mission,"samples":samples,"seed":seed,
 "estimatedReliability":estimate,"analyticReliability":analytic,
 "standardError":se,"confidenceInterval95":ci,"absoluteError":abs_error,
 "acceptanceTolerance":tolerance,"validation":validation,
 "device":device_name,"gpuUsed":gpu_used,"acceleratorUsed":accelerator_used,"engine":selected_engine,
 "backendVersion":backend_version,"elapsedSeconds":elapsed
}
output_digest=hashlib.sha256(canonical_json(result_payload).encode()).hexdigest()
response={
 "protocol":"ayeai.compute.job/v1","jobId":job["jobId"],"inputDigest":job["inputDigest"],
 "outputDigest":output_digest,"providerId":provider_id,"providerVersion":provider_version,
 "status":"SUCCEEDED" if validation=="PASS" else "FAILED",
 "evidenceState":"SIMULATED","result":result_payload,
 "diagnostics":[{"code":"ANALYTIC_CROSSCHECK","message":"Monte Carlo estimate compared with closed-form series reliability."}]
}
assert hashlib.sha256(canonical_json(response["result"]).encode()).hexdigest()==response["outputDigest"]
print(json.dumps(response,indent=2))
with open("/content/compute-result.json","w") as f:
    json.dump(response,f,indent=2,sort_keys=True)
files.download("/content/compute-result.json")


## Interpretation and limitations

- The component lifetimes are independent and exponentially distributed. Common-cause failures, repair, standby redundancy, load sharing, aging and dependencies are not modeled.
- The confidence interval is a normal approximation. It is suitable for this high-sample demonstration; production uncertainty analysis should compare exact/binomial intervals and model uncertainty.
- GPU execution is reported only when `torch.cuda.is_available()` is true. A GPU runtime can be selected in Colab under **Runtime → Change runtime type**.
- A GPU run does not make the result physical evidence. The artifact remains `SIMULATED`.
- Import the JSON result into the host platform only after validating the job ID, input digest, output digest, provider version and configuration binding.
